# Build the IL2CPP Dumper Studio APK on Google Colab
Developed by Mohamed Annati.

Run cells in order. Cell 1 builds the release APK (reuses SDK/Gradle if present). Cell 2 downloads it. Cell 3 (optional) wipes everything for a clean slate.

IMPORTANT: Colab ships Java 21 by default, which breaks AGP's jdk-image transform (`jmod ... --module-version [cgroup warnings]`). This notebook forces JDK 17 and silences the Colab cgroup warnings.

The APK uses the same hexagon + `</>` icon as the web studio and the same on-device dump features.

In [ ]:
%%bash
set -e
cd /content

# 1) Force JDK 17 (Colab's default Java 21 breaks the AGP jdk-image transform)
apt-get update -qq
apt-get install -y -qq openjdk-17-jdk >/dev/null
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
export PATH=$JAVA_HOME/bin:$PATH

# 2) latest source
rm -rf mainproject
git clone -q -b arena/01a0501d-mainproject https://github.com/Mohamed2020p/mainproject

# 3) Android SDK (reuse if already installed)
export ANDROID_HOME=/content/sdk ANDROID_SDK_ROOT=/content/sdk
if [ ! -x sdk/cmdline-tools/latest/bin/sdkmanager ]; then
  mkdir -p sdk/cmdline-tools
  curl -fsSL -o ct.zip https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip
  unzip -q ct.zip -d sdk/cmdline-tools
  mv sdk/cmdline-tools/cmdline-tools sdk/cmdline-tools/latest
  yes | sdk/cmdline-tools/latest/bin/sdkmanager --sdk_root=/content/sdk \
      "platforms;android-34" "build-tools;34.0.0" "platform-tools" >/dev/null
fi

# 4) Gradle 8.7 (reuse if present)
if [ ! -x gradle-8.7/bin/gradle ]; then
  curl -fsSL -o gradle.zip https://services.gradle.org/distributions/gradle-8.7-bin.zip
  unzip -q gradle.zip
fi
export PATH=/content/gradle-8.7/bin:$PATH

# 5) build on JDK 17, silence Colab cgroup warnings, drop poisoned transform cache
cd mainproject/android
echo "sdk.dir=/content/sdk" > local.properties
echo "org.gradle.java.home=/usr/lib/jvm/java-17-openjdk-amd64" >> gradle.properties
echo "org.gradle.jvmargs=-Xmx3072m -Dfile.encoding=UTF-8 -XX:-UseContainerSupport" >> gradle.properties
rm -rf /root/.gradle/caches/transforms-*
gradle --no-daemon assembleRelease

APK=$(find app/build/outputs/apk -name "*.apk" | head -1)
echo "built: $APK"
cp "$APK" /content/IL2CPPDumperStudio.apk
echo "[+] APK ready: /content/IL2CPPDumperStudio.apk"


In [ ]:
from google.colab import files
files.download('/content/IL2CPPDumperStudio.apk')

In [ ]:
# OPTIONAL - wipe EVERYTHING downloaded (SDK, Gradle, source, caches) for a clean slate.
# Only run this if you want to force a full re-download; otherwise the build cell reuses them.
%%bash
rm -rf /content/sdk /content/gradle-8.7 /content/gradle.zip /content/ct.zip
rm -rf /content/mainproject /content/IL2CPPDumperStudio.apk
rm -rf /root/.gradle
echo "[+] cleaned /content and gradle caches"